# Add productUuid and eolCategoryUuid to tBaustoff CSV

This notebook adds new columns to the existing CSV:
- `productUuid`: deterministic UUID per unique combination of `(tBaustoffName, eolCategoryName, eolScenarioReal, eolScenarioPotential, technologyFactor, processCategoryNumber)`
- `eolCategoryUuid`: deterministic UUID per unique combination of `(eolCategoryName, eolScenarioReal, eolScenarioPotential, technologyFactor)`
- `releaseUuid`: constant value `70ee17c1-b144-45d1-97c2-f600f238e112`
- `releaseTag`: constant value `v1.0.0`

Note: UUIDs are generated deterministically using `uuid5` with fixed namespaces so identical combinations always produce the same UUID.


In [5]:
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 5.9 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 8.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pandas]2m3/4 [pandas]


In [ ]:
import pandas as pd
import uuid
import os
import time

# Configuration
INPUT_FILENAME = 'tbaustoff_release_source_data/v1_initial_release_obd_tbaustoff_mapping__70ee17c1-b144-45d1-97c2-f600f238e112.csv'
PRODUCT_UUID_NAMESPACE = uuid.UUID('11111111-1111-1111-1111-111111111111')
EOL_UUID_NAMESPACE = uuid.UUID('22222222-2222-2222-2222-222222222222')

# Define key column combinations
PRODUCT_KEY_COLS = [
    'tBaustoffName',
    'eolCategoryName',
    'eolScenarioReal',
    'eolScenarioPotential',
    'technologyFactor',
    'processCategoryNumber',
]
EOL_KEY_COLS = [
    'eolCategoryName',
    'eolScenarioReal',
    'eolScenarioPotential',
    'technologyFactor',
]

# Helper functions

def make_key_str(row: pd.Series, cols: list[str]) -> str:
    parts = []
    for c in cols:
        v = row.get(c)
        if pd.isna(v):
            parts.append('')
        else:
            parts.append(str(v).strip())
    return '||'.join(parts)


def uuid5_for_key(ns: uuid.UUID, key: str) -> str:
    return str(uuid.uuid5(ns, key))

from pathlib import Path

# Load CSV (works in notebook and script)
try:
    NOTEBOOK_DIR = Path(__file__).parent
except NameError:
    NOTEBOOK_DIR = Path(os.getcwd())

csv_path = (NOTEBOOK_DIR / INPUT_FILENAME).resolve()
df = pd.read_csv(csv_path)
print('Rows loaded:', len(df))

# Compute unique keys
product_keys = df.apply(lambda r: make_key_str(r, PRODUCT_KEY_COLS), axis=1)
eol_keys = df.apply(lambda r: make_key_str(r, EOL_KEY_COLS), axis=1)

# Generate deterministic UUIDs
df['productUuid'] = product_keys.apply(lambda k: uuid5_for_key(PRODUCT_UUID_NAMESPACE, k))
# Generate EOL category UUIDs separately for clarity
df['eolCategoryUuid'] = df.apply(lambda r: uuid5_for_key(EOL_UUID_NAMESPACE, make_key_str(r, EOL_KEY_COLS)), axis=1)


Rows loaded: 588
Written: /Volumes/nextcoder/code/elca/new-app/passport/prisma/seeding/tbaustoff_release_source_data/obd_tbaustoff_mapping_with_uuids_1760452000.csv


In [ ]:

# Output file path
base = os.path.splitext(os.path.basename(INPUT_FILENAME))[0]
ts = int(time.time())
output_filename = f"{base}_with_uuids_{ts}.csv"
output_path = os.path.join(os.path.dirname(csv_path), output_filename)

df.to_csv(output_path, index=False)
print('Written:', output_path)

